文生图

In [12]:

import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

load_dotenv(verbose=True)

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_TEXT_IMG_BASE_URL")

print("api_key:", api_key)

model = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)


def encode_image(image_path):
    # 检查文件存在
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"文件不存在: {image_path}")

    # 检查文件大小
    size = os.path.getsize(image_path)
    if size == 0:
        raise ValueError(f"文件为空: {image_path}")
    print(f"文件大小: {size} 字节")

    with open(image_path, "rb") as f:
        raw = f.read()

    # base64 编码后长度 ≈ 原始大小 * 4/3
    b64 = base64.b64encode(raw).decode("utf-8")
    print(f"Base64 长度: {len(b64)}")
    return b64



# base64_image = encode_image("11.jpeg")
base64_image = encode_image("222.jpg")


print(model.invoke([
HumanMessage(
        content=[
            {"type": "text", "text": """
                这是抖音非常火的gta风格的图片，参照我给你这张图片示例，你给我这张图片风格的提示词，另外再给我以下7个场景的风格提示词（每种风格都需要符合该场景的表情）：
                1. 场景1：桌上时钟 8.50，上班要迟到了
                2. 场景2：在公司工位两台电脑，电脑显示claude code+code的代码正在运行界面，我躺在椅子上玩手机
                3. 场景3：和工位旁边同时聊天，开怀大笑
                4. 场景4：食堂吃饭
                5. 场景5：健身房训练
                6. 场景6：篮球场训练
                7. 场景7：骑着电动车戴着帽子和耳机回家
            """},
            {
                "type": "image_url",
                "image_url": {
                "url": f"data:image/jpeg;base64,{base64_image}"
            }
            }
        ]
    )
]))
print("执行成功")



api_key: None
文件大小: 100293 字节
Base64 长度: 133724
content='# 一、风格拆解（示例图的核心特征）\n\n这张图火爆的关键在于以下几个风格要素，写提示词时必须全部带上：\n\n| 要素 | 对应提示词关键词 |\n|---|---|\n| GTA圣安地列斯游戏画风 | `GTA San Andreas style, Rockstar Games 2004 art style` |\n| PS2时代低模3D渲染 | `PS2-era low-poly 3D render, early 2000s video game graphics` |\n| 夸张搞笑的表情（ meme 脸） | `exaggerated comedic meme facial expression` |\n| 游戏截图质感 | `game screenshot aesthetic, slightly grainy low-res textures` |\n| 柔和偏暖的室内光 | `soft muted indoor lighting, warm color tone` |\n| 生活化道具细节 | `detailed environment props, humorous internet meme vibe` |\n\n## 通用风格模板（可拼接复用）\n\n```text\nGTA San Andreas style 3D render, PS2-era low-poly video game graphics, early 2000s Rockstar Games art style, game screenshot aesthetic, exaggerated comedic meme facial expression, slightly grainy low-resolution textures, soft muted lighting, warm color tone, detailed environment props, humorous internet meme vibe, cinematic close-up composition\n```\n\n> 角色默认描述（保持和示例图同一人物，可替换）：`a young 

图生图

In [13]:

import base64
import mimetypes
from openai import OpenAI

# 1. 编码图片为 Data URL
def encode_image_to_data_url(image_path: str) -> str:
    mime_type, _ = mimetypes.guess_type(image_path)
    if not mime_type or not mime_type.startswith("image/"):
        raise ValueError(f"无法识别的图片格式: {image_path}")
    with open(image_path, "rb") as f:
        b64_data = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime_type};base64,{b64_data}"

base64_image = encode_image_to_data_url("11.jpeg")
print("Data URL 前缀:", base64_image[:50])  # 应输出 data:image/jpeg;base64,...


api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_IMG_TO_IMG_BASE_URL")


# 2. 初始化客户端
client = OpenAI(
    api_key=api_key,
    base_url=base_url
)

# 3. 调用图生图
response = client.images.generate(
    model="qwen-image-3.0-pro",
    prompt="GTA San Andreas style 3D render, PS2-era low-poly video game graphics, early 2000s Rockstar Games art style, office cubicle with two monitors side by side, screens showing Claude Code terminal interface with streaming code, green text and running progress bars on dark background, a young Black man in white tank top slouched deep in the office chair with legs stretched out, lazily scrolling his phone, smug relaxed smirk, half-closed unbothered eyes, office fluorescent lighting, keyboards and coffee cup on desk, game screenshot aesthetic, slightly grainy textures, humorous internet meme vibe",
    size="1024x768",
    n=1,
    extra_body={
        "image": base64_image,      # 必须是完整 Data URL
        "prompt_extend": True,
    },
)

print(response)
print(response.data[0].url)         # 图片 URL，有效期 24 小时

Data URL 前缀: data:image/jpeg;base64,/9j/4AAQSkZJRgABAQAAAQABAAD
ImagesResponse(created=1791283849, background=None, data=[Image(b64_json=None, revised_prompt=None, url='https://dashscope-a717.oss-accelerate.aliyuncs.com/1d/98/20261006/77c155c8/df313a9e-a008-9263-b34a-4162800eb3e4_qwen_image3_serving_output_0.png?Expires=1791370248&OSSAccessKeyId=LTAI5tPxpiCM2hjmWrFXrym1&Signature=ZYn4m3Xf0N4PQ4WuN6RAwKWcO3g%3D')], output_format=None, quality=None, size=None, usage=Usage(input_tokens=None, input_tokens_details=None, output_tokens=None, total_tokens=None, output_tokens_details=None, output_height=768, output_width=1024, input_image_count=1, input_image_type='qima_input_1k', output_image_count=1, output_image_type='qima_output_1k'))
https://dashscope-a717.oss-accelerate.aliyuncs.com/1d/98/20261006/77c155c8/df313a9e-a008-9263-b34a-4162800eb3e4_qwen_image3_serving_output_0.png?Expires=1791370248&OSSAccessKeyId=LTAI5tPxpiCM2hjmWrFXrym1&Signature=ZYn4m3Xf0N4PQ4WuN6RAwKWcO3g%3D
